In [ ]:
# Install geospatial libraries
!pip install geopandas rasterio fiona requests -q
# Test the BC Catalogue API immediately
import requests

response = requests.get(
    "https://catalogue.data.gov.bc.ca/api/3/action/package_search",
    params={"q": "cadastral survey", "rows": 5}
)

data = response.json()
print(f"Success: {data['success']}")
print(f"Total datasets found: {data['result']['count']}")

for dataset in data['result']['results']:
    print(f"\n{dataset['title']}")
    formats = [r['format'] for r in dataset.get('resources', [])]
    print(f"Formats: {formats}")

Success: True
Total datasets found: 35

North Cowichan Cadastral
Formats: ['kmz', 'csv', 'other', 'fgdb', 'gpkg', 'shp']

TANTALIS - Surveyed Parcels
Formats: ['multiple', 'wms', 'kml']

TANTALIS - Integrated Survey Areas
Formats: ['multiple', 'wms', 'kml']

TANTALIS - Surveyed Wellsites
Formats: ['multiple', 'wms', 'kml']

TANTALIS - Surveyed Parcels (Polygons)
Formats: ['oracle_sde']


In [ ]:
import pandas as pd

def explore_datasets(query, rows=50):
    """Pull datasets and display them cleanly for reading"""
    response = requests.get(
        "https://catalogue.data.gov.bc.ca/api/3/action/package_search",
        params={"q": query, "rows": rows}
    )
    data = response.json()
    datasets = data['result']['results']

    records = []
    for ds in datasets:
        formats = [r.get('format', '') for r in ds.get('resources', [])]
        tags = [t['name'] for t in ds.get('tags', [])]

        records.append({
            'title': ds.get('title', ''),
            'notes': ds.get('notes', '')[:300],
            'formats': ', '.join(set(formats)),
            'tags': ', '.join(tags),
            'organization': ds.get('organization', {}).get('title', ''),
            'id': ds.get('name', '')
        })

    df = pd.DataFrame(records)
    return df

# Run several different searches
df_survey = explore_datasets("survey", rows=50)
df_cadastral = explore_datasets("cadastral", rows=50)
df_parcel = explore_datasets("parcel", rows=50)
df_terrain = explore_datasets("elevation terrain lidar", rows=50)

# Combine them
df_all = pd.concat([df_survey, df_cadastral, df_parcel, df_terrain])
df_all = df_all.drop_duplicates(subset='id')

print(f"Total unique datasets to review: {len(df_all)}")
df_all.head(20)
df_all.to_csv('bc_datasets_explore.csv', index=False)
print("CSV saved!")


Total unique datasets to review: 130
CSV saved!


LABEL AS 1 (relevant) IF:
- Title or description mentions: survey, cadastral, parcel,
  boundary, monument, benchmark, legal plan, property,
  land title, tenure, easement, right-of-way
- AND data represents actual geographic features
  (not administrative or statistical data)

LABEL AS 0 (not relevant) IF:
- About population, health, social services, weather
- Administrative boundaries only (no survey content)
- Imagery without survey attributes
- Aggregated statistical data

